In [1]:
from fem import (
    # Gmsh tools
    GMSHtools,
    # Visualization — Gmsh
    add_element_data_view, add_node_data_view, compute_nodal_average,
    results2gmsh,
    # Plotting — matplotlib
    plot_gmsh_mesh,opensees2gmsh,
    # Parameters
    globalParameters,
)

import os
import numpy as np
import matplotlib.pyplot as plt
import gmsh

np.set_printoptions(suppress=True, precision=6, linewidth=400)


  FEM -- Finite Element Method for Structural Analysis
  Based on the course by Prof. José Abell

  Version 1.1.0                        © 2026 All Rights Reserved

  Repository  :  https://github.com/ppalacios92/FEM
  Web Book    :  https://books.nmorabowen.com/books/fem

  Patricio Palacios B.    |    Nicolas Mora Bowen

  ********* (>'-')> Ladruño4ever  *********



In [2]:
globalParameters['nDoF'] = 3
globalParameters['nDIM'] = 3

In [3]:
# General model parameters
output_file = 'example_PP1.msh'

In [4]:
# read mesh — node_map and system_nDof auto-generated
mesh = GMSHtools(output_file)

  MESH SUMMARY

  === NODES ===  (6834 total — showing first 3)
     Tag              x              y              z
--------------------------------------------------------------------------------
       1         0.0000         0.0000         0.0000
       2      5000.0000         0.0000         0.0000
       3      5000.0000       300.0000         0.0000
--------------------------------------------------------------------------------

  === PHYSICAL GROUPS ===  (4 total)
      ID    Dim   Name
--------------------------------------------------------------------------------
      21      1   'support_xy'
      22      1   'support_y'
      23      2   'load'
      24      3   'solid'
--------------------------------------------------------------------------------

  === ELEMENTS ===  (4 groups)
      ID    Dim     Type   Nodes/el   N elements   Name
--------------------------------------------------------------------------------
      21      1        1          2            6   'su

In [5]:

load_dictionary = {
                23:   {'value': 1, 'direction': '-z'},     
}

In [6]:
# build lumped nodal force vector
F_nodal = mesh.build_load_vector(load_dictionary)

# assemble to global vector
F_load = np.zeros(mesh.system_nDof)
for tag, f_vec in F_nodal.items():
    if tag in mesh.node_map:
        F_load[mesh.node_map[tag].idx[:len(f_vec)]] += f_vec
F_load[np.abs(F_load) < 1e-4] = 0.0

In [7]:
# # %matplotlib widget

# plot_gmsh_mesh(mesh,
#                show_node_labels   = False,
#                show_element_labels= False,
#                show_node_points   = False,
#                view_3d            = True,   elev= 45, azim= -45,
#                figsize            = (12, 8))

## Opensees

In [8]:
import opensees as ops
import opsvis as opsv

ops.wipe()
ops.model('basicBuilder', '-ndm', 3, '-ndf', 3)



         OpenSees -- Open System For Earthquake Engineering Simulation
                 Pacific Earthquake Engineering Research Center
                        Version 3.7.2 64-Bit


      (c) Copyright 1999-2016 The Regents of the University of California
                              All Rights Reserved
  (Copyright and Disclaimer @ http://www.berkeley.edu/OpenSees/copyright.html)


  *********** (o_O) OPENSEES (>'-')> Ladruno4ever *********** 3.7.2




In [9]:
# Nodes
for tag, (x, y, z) in mesh.nodes.items():
    ops.node(tag, x, y , z)

In [10]:
# Boundary conditions
fixed_nodes = set()
for tag in mesh.physical_groups['support_xy'].nodes:
    if tag not in fixed_nodes:
        fixed_nodes.add(tag)
        # ops.fix(tag, 1, 1, 1, 1, 1, 1)
        ops.fix(tag, 1, 1, 1)

In [11]:
# Boundary conditions
fixed_nodes = set()
for tag in mesh.physical_groups['support_y'].nodes:
    if tag not in fixed_nodes:
        fixed_nodes.add(tag)
        # ops.fix(tag, 1, 1, 1, 1, 1, 1)
        ops.fix(tag, 0, 1, 1)

In [12]:
# Material
E = 3500      
nu = 0.36     
rho = 2400e-9  
ops.nDMaterial('ElasticIsotropic', 1, E, nu, rho)

In [13]:
group = mesh.physical_groups['solid'].elements
for elem_tag, conn in zip(group['element_tags'], group['connectivity']):
    n1, n2, n3, n4 = conn
    ops.element('FourNodeTetrahedron', elem_tag, n1, n2, n3, n4, 1)

# group = mesh.physical_groups['solid'].elements
# for elem_tag, conn in zip(group['element_tags'], group['connectivity']):
#     ops.element('FourNodeTetrahedron', elem_tag, *conn, 1)
    

In [14]:
ops.timeSeries('Linear', 1)
ops.pattern('Plain', 1, 1)
for tag, force in F_nodal.items():
    if np.any(np.abs(force) > 0):
        ops.load(tag, *force.tolist())

## eig values

In [15]:
node_tags = np.array(list(mesh.nodes.keys()))
n_nodes   = len(mesh.nodes)

# Eigenanalysis
n_modes = 6
eigenvalues = ops.eigen(n_modes)
omega = np.sqrt(np.array(eigenvalues))
freq  = omega / (2 * np.pi)
period = 1 / freq

print(f"{'Mode':>6} {'Freq [Hz]':>12} {'Period [s]':>12}")
for i in range(n_modes):
    print(f"{i+1:>6} {freq[i]:>12.4f} {period[i]:>12.4f}")



  Mode    Freq [Hz]   Period [s]
     1       0.2635       3.7952
     2       0.3354       2.9813
     3       0.8782       1.1387
     4       1.0743       0.9308
     5       1.1767       0.8498
     6       1.6111       0.6207


## Animation

In [16]:
mode    = 2
n_steps = 30
factor  = 5000

mode_3d = np.zeros((n_nodes, 3))
for i, tag in enumerate(mesh.nodes.keys()):
    mode_3d[i, 0] = ops.nodeEigenvector(tag, mode, 1)
    mode_3d[i, 1] = ops.nodeEigenvector(tag, mode, 2)
    mode_3d[i, 2] = ops.nodeEigenvector(tag, mode, 3)

gmsh.initialize()
gmsh.open(output_file)

view_mode = gmsh.view.add(f"Mode {mode} — T={period[mode-1]:.4f}s")

for step in range(n_steps):
    scale = np.cos(step / n_steps * 2 * np.pi)
    gmsh.view.addHomogeneousModelData(
        tag           = view_mode,
        step          = step,
        time          = float(step),
        modelName     = gmsh.model.getCurrent(),
        dataType      = "NodeData",
        numComponents = -1,
        tags          = node_tags,
        data          = (mode_3d * scale).reshape(-1)
    )

gmsh.view.option.setNumber(view_mode, "VectorType",         5)
gmsh.view.option.setNumber(view_mode, "DisplacementFactor", factor)
gmsh.fltk.run()
gmsh.finalize()

## Vertical load

In [17]:
# NstepGravity=10
# DGravity=1/NstepGravity

# ops.system("FullGeneral")
# ops.numberer("Plain")
# ops.constraints("Plain")
# ops.integrator("LoadControl", DGravity )
# ops.test("NormUnbalance", 1.0e-6, 100 , 0)
# ops.algorithm("Newton")
# ops.analysis("Static")

# node_tags = np.array(list(mesh.nodes.keys()))
# n_nodes   = len(mesh.nodes)

# gmsh.initialize()
# gmsh.open(output_file)

# view_disp = gmsh.view.add("Displacements")

# for step in range(NstepGravity):
#     ops.analyze(1)

#     u_3d = np.zeros((n_nodes, 3))
#     for i, tag in enumerate(mesh.nodes.keys()):
#         u_3d[i, 0] = ops.nodeDisp(tag, 1)
#         u_3d[i, 1] = ops.nodeDisp(tag, 2)
#         u_3d[i, 2] = ops.nodeDisp(tag, 3)

#     gmsh.view.addHomogeneousModelData(
#         tag=view_disp, step=step, time=float(step),
#         modelName=gmsh.model.getCurrent(),
#         dataType="NodeData", numComponents=-1,
#         tags=node_tags, data=u_3d.reshape(-1))

# gmsh.view.option.setNumber(view_disp, "VectorType",          5)
# gmsh.view.option.setNumber(view_disp, "DisplacementFactor",  10)

# gmsh.fltk.run()
# gmsh.finalize()

## Vertical Analysis

In [ ]:
NstepGravity=10
DGravity=1/NstepGravity

ops.system("BandGeneral") 
ops.numberer("RCM")
ops.constraints("Plain")
ops.integrator("LoadControl", DGravity )
ops.test("NormUnbalance", 1.0e-6, 100 , 0)
ops.algorithm("Newton")
ops.analysis("Static")

ops.analyze(NstepGravity)


In [19]:

opensees2gmsh(
    output_file      = output_file,
    mesh             = mesh,
    ops              = ops,
    solid_group_name = 'solid',
    F_nodal          = F_nodal,
    # nDOF             = 3,
    disp_factor      = 2,
    show_disp        = True,
    show_loads       = True,
    show_reactions   = True,
    show_stress      = True,
    show_vm          = True,
    show_averaged    = True,
)